# __Clean Data__

## __Evaluación__

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv('../data/raw/eval.csv',sep=';')


data.head()

,FDS,Nombre del Producto,Fabricante,Uso,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,...,Item_14_6,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1
0,2,PRODUCTO 1 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,20/09/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
1,4,PRODUCTO 2 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
2,6,PRODUCTO 3 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
3,25,PRODUCTO 4 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,4/01/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
4,27,PRODUCTO 5 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,5/06/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR


In [2]:
data["FDS"] = data["FDS"].astype("Int64")
data.head()

,FDS,Nombre del Producto,Fabricante,Uso,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,...,Item_14_6,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1
0,2,PRODUCTO 1 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,20/09/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
1,4,PRODUCTO 2 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
2,6,PRODUCTO 3 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
3,25,PRODUCTO 4 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,4/01/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR
4,27,PRODUCTO 5 - INDUSTRIAL - FABRICANTE 1,FABRICANTE 1,Industrial,5/06/20,Presente,NO_Conf,Presente,Conf_CR,Presente,...,Presente,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR


In [3]:
import pandas as pd

col = "Nombre del Producto"  # ajusta si tu columna se llama distinto

s = (
    data[col]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)   # aplana saltos de línea/espacios
)

# Extrae números
prod = pd.to_numeric(s.str.extract(r"PRODUCTO\s*(\d+)", expand=False), errors="coerce")
fab  = pd.to_numeric(s.str.extract(r"FABRICANTE\s*(\d+)", expand=False), errors="coerce")

# Tipo: I si contiene INDUSTRIAL, si no -> D (puedes hacerlo más estricto si quieres)
tipo = s.str.contains(r"\bINDUSTRIAL\b", regex=True).map({True: "I", False: "D"})

# Construye el id
data["id"] = (
    "F" + fab.astype("Int64").astype(str).str.zfill(2) +
    "-P" + prod.astype("Int64").astype(str).str.zfill(2) +
    "-" + tipo
)

# Correcciones confirmadas al revisar las FDS originales.
# El archivo bruto se conserva como registro de auditoría.
correcciones_presencia = {
    # F06: el contenido de uso recomendado/restricciones está presente,
    # aunque la FDS no usa el subtítulo recomendado por el SGA.
    ("F06-P01-I", "Item_1_3"): "Presente",
    ("F06-P02-I", "Item_1_3"): "Presente",
    ("F06-P01-D", "Item_1_3"): "Presente",
    ("F06-P03-I", "Item_1_3"): "Presente",
    ("F06-P02-D", "Item_1_3"): "Presente",
    ("F06-P03-D", "Item_1_3"): "Presente",
    ("F06-P04-I", "Item_1_3"): "Presente",
    ("F06-P04-D", "Item_1_3"): "Presente",
    ("F06-P05-D", "Item_1_3"): "Presente",
    ("F06-P05-I", "Item_1_3"): "Presente",
    # En 6.2, 8.2 y 11.4 una calidad positiva implica que se observó
    # al menos un atributo; por tanto, la presencia debe ser positiva.
    ("F01-P01-I", "Item_6_2"): "Presente",
    ("F01-P02-I", "Item_6_2"): "Presente",
    ("F01-P03-I", "Item_6_2"): "Presente",
    ("F02-P01-D", "Item_6_2"): "Presente",
    ("F02-P02-D", "Item_6_2"): "Presente",
    ("F02-P04-D", "Item_8_2"): "Presente",
    ("F02-P05-D", "Item_8_2"): "Presente",
    ("F02-P04-D", "Item_11_4"): "Presente",
    ("F02-P05-D", "Item_11_4"): "Presente",
    # Una calidad Conf_CR en 1.2 o 2.2 requiere que exista contenido
    # identificable, aunque incompleto o impreciso.
    ("F03-P02-I", "Item_1_2"): "Presente",
    ("F02-P02-D", "Item_2_2"): "Presente",
    ("F01-P01-I", "Item_7_2"): "Presente",
    ("F01-P02-I", "Item_7_2"): "Presente",
    ("F02-P03-D", "Item_14_5"): "Presente",
    ("F02-P04-D", "Item_14_5"): "Presente",
    ("F02-P05-D", "Item_14_5"): "Presente",
}

for (id_fds, columna), valor in correcciones_presencia.items():
    seleccion = data["id"].eq(id_fds)
    assert seleccion.sum() == 1, f"Identificador no único o ausente: {id_fds}"
    data.loc[seleccion, columna] = valor

# Verificación directa de F03-P03-D: el ítem 1.2 está ausente y su
# calificación correcta es 1 (no confiable), no 5 (confiable).
seleccion_1_2 = data["id"].eq("F03-P03-D")
assert seleccion_1_2.sum() == 1
assert data.loc[seleccion_1_2, "Item_1_2"].astype(str).str.strip().str.lower().eq("no present").all()
assert data.loc[seleccion_1_2, "Calidad_1_2"].eq("Confiable").all()
data.loc[seleccion_1_2, "Calidad_1_2"] = "NO_Conf"

assert data.loc[data["id"].isin(["F01-P01-I", "F01-P02-I"]), "Calidad_7_2"].eq("Confiable").all()
assert data.loc[data["id"].isin(["F02-P03-D", "F02-P04-D", "F02-P05-D"]), "Calidad_14_5"].eq("Conf_CR").all()
ids_f06_item_1_3 = [id_fds for id_fds, columna in correcciones_presencia if columna == "Item_1_3"]
assert len(ids_f06_item_1_3) == 10
assert data.loc[data["id"].isin(ids_f06_item_1_3), "Item_1_3"].eq("Presente").all()
assert data.loc[data["id"].isin(ids_f06_item_1_3), "Calidad_1_3"].eq("Confiable").all()
for item, ids, calificaciones in [
    ("6_2", ["F01-P01-I", "F01-P02-I", "F01-P03-I", "F02-P01-D", "F02-P02-D"], {"Confiable", "Conf_CR"}),
    ("8_2", ["F02-P04-D", "F02-P05-D"], {"Conf_CR"}),
    ("11_4", ["F02-P04-D", "F02-P05-D"], {"Conf_CR"}),
]:
    seleccion = data["id"].isin(ids)
    assert seleccion.sum() == len(ids)
    assert data.loc[seleccion, f"Item_{item}"].eq("Presente").all()
    assert set(data.loc[seleccion, f"Calidad_{item}"]).issubset(calificaciones)
assert data.loc[data["id"].eq("F03-P02-I"), "Item_1_2"].eq("Presente").all()
assert data.loc[data["id"].eq("F03-P02-I"), "Calidad_1_2"].eq("Conf_CR").all()
assert data.loc[data["id"].eq("F02-P02-D"), "Item_2_2"].eq("Presente").all()
assert data.loc[data["id"].eq("F02-P02-D"), "Calidad_2_2"].eq("Conf_CR").all()

# Invariante de coherencia entre presencia y calidad. Las únicas
# combinaciones positivas con ausencia admitidas corresponden
# únicamente a la excepción de confidencialidad de S3.
ausente_calidad_positiva = []
for columna_item in [c for c in data.columns if c.startswith("Item_")]:
    sufijo = columna_item.removeprefix("Item_")
    columna_calidad = f"Calidad_{sufijo}"
    if columna_calidad not in data.columns:
        continue
    mascara = (
        data[columna_item].astype(str).str.strip().str.lower().eq("no present")
        & data[columna_calidad].isin(["Confiable", "Conf_CR"])
    )
    ausente_calidad_positiva.extend(
        (id_fds, sufijo) for id_fds in data.loc[mascara, "id"]
    )

casos_confidencialidad = [
    caso for caso in ausente_calidad_positiva
    if caso[1] in {"3_2_1_1", "3_2_1_2", "3_2_1_3"}
]
assert len(ausente_calidad_positiva) == 29
assert len(casos_confidencialidad) == 29
assert len(casos_confidencialidad) == len(ausente_calidad_positiva)

data["id"].head(51)

# 1) borrar la columna
data = data.drop(columns=["Nombre del Producto"])


In [4]:
data.head()

,FDS,Fabricante,Uso,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,Calidad_1_2,...,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1,id
0,2,FABRICANTE 1,Industrial,20/09/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P01-I
1,4,FABRICANTE 1,Industrial,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P02-I
2,6,FABRICANTE 1,Industrial,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P03-I
3,25,FABRICANTE 1,Industrial,4/01/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P04-I
4,27,FABRICANTE 1,Industrial,5/06/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P05-I


In [5]:
# 1) Cambiar nombre de columna: Fabricante -> fabricante
data = data.rename(columns={"Fabricante": "fabricante"})

# 2) En 'fabricante': cambiar "FABRICANTE X" -> "FX"
#    (soporta espacios variables y mayúsc/minúsc; deja NaN intactos)
data["fabricante"] = (
    data["fabricante"]
    .astype(str)
    .str.upper()
    .str.replace(r"\s+", " ", regex=True)
    .str.replace(r"^FABRICANTE\s*(\d+)$", r"F\1", regex=True)
)

data["fabricante"] = data["fabricante"].str.replace(r"^F(\d+)$", lambda m: f"F{int(m.group(1)):02d}", regex=True)

# 3) En Uso: renombrar columna a USO y pasar valores a mayúscula
data = data.rename(columns={"Uso": "USO"})
data["USO"] = data["USO"].astype(str).str.upper()


In [6]:
data.head()

,FDS,fabricante,USO,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,Calidad_1_2,...,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1,id
0,2,F01,INDUSTRIAL,20/09/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P01-I
1,4,F01,INDUSTRIAL,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P02-I
2,6,F01,INDUSTRIAL,28/02/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P03-I
3,25,F01,INDUSTRIAL,4/01/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P04-I
4,27,F01,INDUSTRIAL,5/06/20,Presente,NO_Conf,Presente,Conf_CR,Presente,Conf_CR,...,Confiable,Presente,Confiable,Presente,NO_Conf,Presente,Confiable,Presente,Conf_CR,F01-P05-I


In [7]:
import numpy as np

mapa = {
    "presente": "PRESENTE",
    "no present": "AUSENTE",
}

def normaliza_valor(x):
    if isinstance(x, str):
        k = " ".join(x.strip().lower().split())  # quita espacios extra y normaliza
        return mapa.get(k, x)                    # solo cambia si coincide
    return x

data = data.apply(lambda col: col.map(normaliza_valor))

In [8]:
data.head()

,FDS,fabricante,USO,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,Calidad_1_2,...,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1,id
0,2,F01,INDUSTRIAL,20/09/20,PRESENTE,NO_Conf,PRESENTE,Conf_CR,PRESENTE,Conf_CR,...,Confiable,PRESENTE,Confiable,PRESENTE,NO_Conf,PRESENTE,Confiable,PRESENTE,Conf_CR,F01-P01-I
1,4,F01,INDUSTRIAL,28/02/20,PRESENTE,NO_Conf,PRESENTE,Conf_CR,PRESENTE,Conf_CR,...,Confiable,PRESENTE,Confiable,PRESENTE,NO_Conf,PRESENTE,Confiable,PRESENTE,Conf_CR,F01-P02-I
2,6,F01,INDUSTRIAL,28/02/20,PRESENTE,NO_Conf,PRESENTE,Conf_CR,PRESENTE,Conf_CR,...,Confiable,PRESENTE,Confiable,PRESENTE,NO_Conf,PRESENTE,Confiable,PRESENTE,Conf_CR,F01-P03-I
3,25,F01,INDUSTRIAL,4/01/20,PRESENTE,NO_Conf,PRESENTE,Conf_CR,PRESENTE,Conf_CR,...,Confiable,PRESENTE,Confiable,PRESENTE,NO_Conf,PRESENTE,Confiable,PRESENTE,Conf_CR,F01-P04-I
4,27,F01,INDUSTRIAL,5/06/20,PRESENTE,NO_Conf,PRESENTE,Conf_CR,PRESENTE,Conf_CR,...,Confiable,PRESENTE,Confiable,PRESENTE,NO_Conf,PRESENTE,Confiable,PRESENTE,Conf_CR,F01-P05-I


In [9]:
mapa2 = {
    "confiable": "CONFIABLE",
    "no_conf": "NO_CONFIABLE",
    "conf_cr": "CONFI_RESTR",
}

def normaliza_valor_2(x):
    if isinstance(x, str):
        k = " ".join(x.strip().lower().split())
        return mapa2.get(k, x)
    return x

cols_texto = data.select_dtypes(include=["object", "string"]).columns
data[cols_texto] = data[cols_texto].apply(lambda col: col.map(normaliza_valor_2))


In [10]:
data.head()

,FDS,fabricante,USO,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,Calidad_1_2,...,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1,id
0,2,F01,INDUSTRIAL,20/09/20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P01-I
1,4,F01,INDUSTRIAL,28/02/20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P02-I
2,6,F01,INDUSTRIAL,28/02/20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P03-I
3,25,F01,INDUSTRIAL,4/01/20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P04-I
4,27,F01,INDUSTRIAL,5/06/20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P05-I


In [11]:
data["Fecha Elaboración"] = pd.to_datetime(
    data["Fecha Elaboración"],
    format="%d/%m/%y",
    dayfirst=True
)
data.head()

,FDS,fabricante,USO,Fecha Elaboración,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,Calidad_1_2,...,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1,id
0,2,F01,INDUSTRIAL,2020-09-20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P01-I
1,4,F01,INDUSTRIAL,2020-02-28,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P02-I
2,6,F01,INDUSTRIAL,2020-02-28,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P03-I
3,25,F01,INDUSTRIAL,2020-01-04,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P04-I
4,27,F01,INDUSTRIAL,2020-06-05,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P05-I


In [12]:
data = data.rename(columns={
    "Fecha Elaboración": "elaborada",
    "USO": "uso",
})

In [13]:
data.head()

,FDS,fabricante,uso,elaborada,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,Calidad_1_2,...,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1,id
0,2,F01,INDUSTRIAL,2020-09-20,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P01-I
1,4,F01,INDUSTRIAL,2020-02-28,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P02-I
2,6,F01,INDUSTRIAL,2020-02-28,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P03-I
3,25,F01,INDUSTRIAL,2020-01-04,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P04-I
4,27,F01,INDUSTRIAL,2020-06-05,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,CONFI_RESTR,...,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR,F01-P05-I


In [14]:
primeras = ["FDS", "elaborada", "id", "fabricante", "uso"]

data = data[primeras + [c for c in data.columns if c not in primeras]]


In [15]:
data.head()

,FDS,elaborada,id,fabricante,uso,Item_1_0,Calidad_1_0,Item_1_1,Calidad_1_1,Item_1_2,...,Item_14_6,Calidad_14_6,Item_15_0,Calidad_15_0,Item_15_1,Calidad_15_1,Item_16_0,Calidad_16_0,Item_16_1,Calidad_16_1
0,2,2020-09-20,F01-P01-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
1,4,2020-02-28,F01-P02-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
2,6,2020-02-28,F01-P03-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
3,25,2020-01-04,F01-P04-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
4,27,2020-06-05,F01-P05-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR


In [16]:
import re

def normaliza_col(c):
    c2 = c.strip()
    c2 = re.sub(r"^Item_", "ITEM_", c2, flags=re.IGNORECASE)
    c2 = re.sub(r"^Calidad_", "CALIDAD_", c2, flags=re.IGNORECASE)
    return c2.upper()

data = data.rename(columns={c: normaliza_col(c) for c in data.columns})


In [17]:
data.head()

,FDS,ELABORADA,ID,FABRICANTE,USO,ITEM_1_0,CALIDAD_1_0,ITEM_1_1,CALIDAD_1_1,ITEM_1_2,...,ITEM_14_6,CALIDAD_14_6,ITEM_15_0,CALIDAD_15_0,ITEM_15_1,CALIDAD_15_1,ITEM_16_0,CALIDAD_16_0,ITEM_16_1,CALIDAD_16_1
0,2,2020-09-20,F01-P01-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
1,4,2020-02-28,F01-P02-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
2,6,2020-02-28,F01-P03-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
3,25,2020-01-04,F01-P04-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
4,27,2020-06-05,F01-P05-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR


In [18]:
# Columnas a modificar
cols_to_set = [
    "ITEM_3_1_1", "CALIDAD_3_1_1",
    "ITEM_3_1_1_1", "CALIDAD_3_1_1_1",
    "ITEM_3_1_1_2", "CALIDAD_3_1_1_2",
    "ITEM_3_1_1_3", "CALIDAD_3_1_1_3",
]

existing = [c for c in cols_to_set if c in data.columns]

# 1) Forzar dtype object columna a columna (robusto)
for c in existing:
    data[c] = data[c].astype("object")

# 2) Asignar NO_APLICA
for c in existing:
    data.loc[:, c] = "NO_APLICA"

# 3) Eliminar CALIDAD_3_5
data.drop(columns=["CALIDAD_3_5"], inplace=True, errors="ignore")

In [19]:
data.head()

,FDS,ELABORADA,ID,FABRICANTE,USO,ITEM_1_0,CALIDAD_1_0,ITEM_1_1,CALIDAD_1_1,ITEM_1_2,...,ITEM_14_6,CALIDAD_14_6,ITEM_15_0,CALIDAD_15_0,ITEM_15_1,CALIDAD_15_1,ITEM_16_0,CALIDAD_16_0,ITEM_16_1,CALIDAD_16_1
0,2,2020-09-20,F01-P01-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
1,4,2020-02-28,F01-P02-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
2,6,2020-02-28,F01-P03-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
3,25,2020-01-04,F01-P04-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR
4,27,2020-06-05,F01-P05-I,F01,INDUSTRIAL,PRESENTE,NO_CONFIABLE,PRESENTE,CONFI_RESTR,PRESENTE,...,PRESENTE,CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,NO_CONFIABLE,PRESENTE,CONFIABLE,PRESENTE,CONFI_RESTR


In [20]:
data.to_csv('../data/processed/eval.csv', index=False)

## __Diccionario de Datos__

In [21]:
import re
import pandas as pd

texto = r"""
1.0. 	Título: Identificación del producto
1.1.	Identificador SGA del producto
1.2.	Otros medios de identificación
1.3.	Uso recomendado del producto químico y restricciones de uso
1.4.	Datos del proveedor 
1.5.	Número de teléfono para emergencias 
2.0. 	Título: Identificación del peligro o peligros
2.1.	Clasificación  SGA  de  la  sustancia/mezcla  y  cualquier  información nacional o regional
2.1.1. 	Elementos de la etiqueta SGA: Pictogramas
2.1.2. 	Elementos de la etiqueta SGA: Palabra de Advertencia
2.1.3. 	Elementos de la etiqueta SGA: Indicaciones de Peligro
2.1.4. 	Elementos de la etiqueta SGA Consejos de Prudencia
2.2. 	Otros peligros que no figuren en la clasificación (por ejemplo, peligro de explosión de partículas de polvo) o que no están cubiertos por el SGA
3.0. 	Título: Composición/información sobre los componentes
3.1.1. 	Identidad química de la sustancia 
3.1.1.1.	Nombres comunes o sinónimos de la sustancia 
3.1.1.2.	Número CAS y otros identificadores únicos de la sustancia 
3.1.1.3.	Impurezas, aditivos o estabilizadores presentes en la sustancia 
3.2.1. 	Identidad química de la sustancia 
3.2.1.1.	Nombres comunes o sinónimos de la mezcla
3.2.1.2.	Número CAS y otros identificadores únicos de la mezcla
3.2.1.3. 	Concentración o gamas de concentraciones de todos los componentes de la mezcla 
4.0. 	Título: Primeros auxilios
4.1.	Descripción de las medidas necesarias, desglosadas con arreglo a las diferentes vías de exposición, esto es, inhalación, contacto cutáneo y ocular e ingestión
4.2.	Síntomas/efectos más importantes, agudos y retardados
4.3.	Indicación de la necesidad de recibir atención médica inmediata y tratamiento especial requerido en caso necesario
5.0.	Título: Medidas de lucha contra incendios
5.1.	Medios adecuados (o no adecuados) de extinción
5.2.	Peligros específicos del producto químico
5.3.	Medidas especiales que deben tomar los equipos de lucha contra incendios
6.0.	Título: Medidas que deben tomarse en caso de vertido accidental
6.1.	Precauciones individuales, equipos de protección y procedimientos de emergencia
6.2.	Precauciones relativas al medio ambiente 
6.3.	Métodos y materiales para la contención y limpieza de vertidos
7.0.	Título: Manipulación y almacenamiento
7.1.	Precauciones que se deben tomar para garantizar una manipulación segura
7.2.	Condiciones   de   almacenamiento   seguro,   incluidas   cualesquiera incompatibilidades
8.0.	Controles de exposición/protección personal
8.1.	Parámetros  de  control:  límites  o  valores  de  corte  de  exposición ocupacionales o biológicos
8.2.	Controles técnicos apropiados
8.3.	Medidas  de  protección  individual,  como  equipos  de  protección personal
9.0.	Título: Propiedades físicas y químicas
9.1. 	Propiedades fisicas y químicas 
10.0.	Título: Estabilidad y reactividad
10.1. 	Reactividad 
10.2. 	Estabilidad química 
10.3. 	Posibilidad de reacciones peligrosas
10.4. 	Condiciones que deben evitarse 
10.5. 	Materiales incompatibles 
10.6. 	Productos de descomposición peligrosos
11.0.	Título: Información toxicológica 
11.1. 	Efectos toxicológicos 
11.2. 	Información sobre las posibles vías de exposición 
11.3. 	Síntomas relacionados con las características físicas, químicas y toxicológicas 
11.4.	Efectos inmediatos y retardados así como efectos crónicos producidos por una exposición a corto y largo plazo
11.5. 	Medidas numéricas de toxicidad (tales como estimaciones de toxicidad aguda) 
12.0.	Título: Información ecotoxicológica
12.1. 	Toxicidad 
12.2. 	Persistencia y degradabilidad 
12.3. 	Potencial de bioacumulación 
12.4.	Movilidad en el suelo
12.5. 	Otros efectos adversos
13.0.	Título: Información relativa a la eliminación de los productos 
13.1. 	Métodos de eliminación 
14.0.	Título: Información relativa al transporte 
14.1. 	Número ONU 
14.2. 	Designación oficial de transporte de las Naciones Unidas
14.3. 	Clase(s) relativas al transporte
14.4. 	Grupo de embalaje/envasado si se aplica
14.5. 	Riesgos ambientales
14.6. 	Precauciones especiales para el usuario 
15.0.	Título: Información sobre la reglamentación
15.1. 	Disposiciones específicas sobre seguridad, salud y medio ambiente para el producto de que se trate
16.0.	Título: Otras informaciones 
16.1. 	Otras informaciones 
"""

pat = re.compile(r"^\s*(\d+(?:\.\d+)+)\.\s*(.+?)\s*$")

rows = []
for line in texto.splitlines():
    line = line.strip()
    if not line:
        continue
    m = pat.match(line)
    if not m:
        continue

    codigo = m.group(1)                 # ej: "3.1.1.2"
    desc = m.group(2).strip()           # texto
    key = codigo.replace(".", "_")      # ej: "3_1_1_2"

    rows.append({"seccion": key, "descripcion": desc})

df_diccionario = pd.DataFrame(rows)
df_diccionario



,seccion,descripcion
0,1_0,Título: Identificación del producto
1,1_1,Identificador SGA del producto
2,1_2,Otros medios de identificación
3,1_3,Uso recomendado del producto químico y restric...
4,1_4,Datos del proveedor
...,...,...
70,14_6,Precauciones especiales para el usuario
71,15_0,Título: Información sobre la reglamentación
72,15_1,"Disposiciones específicas sobre seguridad, sal..."
73,16_0,Título: Otras informaciones


In [22]:
import numpy as np
import pandas as pd

df = df_diccionario.copy()

# Ordena por sección/subsección numérica
tmp = df["seccion"].str.split("_", expand=True)
df["_sec"] = pd.to_numeric(tmp[0], errors="coerce")
df["_sub"] = pd.to_numeric(tmp[1], errors="coerce")
df = df.sort_values(["_sec", "_sub"]).reset_index(drop=True)

# Filas de título (sub == 0)
is_titulo = df["_sub"].eq(0)

# Texto limpio del título (sin "Título:")
titulo_limpio = (
    df["descripcion"].astype(str)
    .str.replace(r"^\s*T[ÍI]TULO\s*:\s*", "", regex=True)
    .str.strip()
)

# Nombre de sección propagado hacia abajo
df["seccion_nombre"] = np.where(is_titulo, titulo_limpio, np.nan)
df["seccion_nombre"] = df["seccion_nombre"].ffill()

# Subsección: vacío para los _0; texto para el resto
df["subseccion"] = np.where(is_titulo, "", df["descripcion"].astype(str).str.strip())

# Resultado final
df_final = (
    df.rename(columns={"seccion": "id", "seccion_nombre": "seccion"})
      [["id", "seccion", "subseccion"]]
)

df_final

# ... (todo igual hasta df_final)

df_final = (
    df.rename(columns={"seccion": "id", "seccion_nombre": "seccion"})
      [["id", "seccion", "subseccion"]]
)

df_final["seccion"] = (
    df_final["seccion"].astype(str)
    .str.replace("Título:", "", regex=True)
    .str.strip()
)

# 1) Renombrar columna "subseccion" -> "descripcion"
df_final = df_final.rename(columns={"subseccion": "descripcion"})

# 2) Para ids que terminan en "_0", poner "Título de sección" en descripcion
mask_titulo = df_final["id"].astype(str).str.endswith("_0")
df_final.loc[mask_titulo, "descripcion"] = "Título de sección"


df_final

,id,seccion,descripcion
0,1_0,Identificación del producto,Título de sección
1,1_1,Identificación del producto,Identificador SGA del producto
2,1_2,Identificación del producto,Otros medios de identificación
3,1_3,Identificación del producto,Uso recomendado del producto químico y restric...
4,1_4,Identificación del producto,Datos del proveedor
...,...,...,...
70,14_6,Información relativa al transporte,Precauciones especiales para el usuario
71,15_0,Información sobre la reglamentación,Título de sección
72,15_1,Información sobre la reglamentación,"Disposiciones específicas sobre seguridad, sal..."
73,16_0,Otras informaciones,Título de sección


In [23]:
df_final.to_csv('../data/processed/diccionario_items.csv', index=False)